In [1]:
import pandas as pd

## Import slimmed down Gold-Standard

In [39]:
gs = pd.read_csv("../gs_slim.csv")

In [48]:
gs["scope"].unique()

<StringArray>
['1', '2lb', '2mb', '3']
Length: 4, dtype: str

## Import all 3x2=6 Extractions

In [16]:
think1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Thinking.csv")
think2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Thinking.csv")

moe1   = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-30B-A3B-Thinking.csv")
moe2   = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-30B-A3B-Thinking.csv")

instr1 = pd.read_csv("./PipelineB-Answers/1st_Qwen3-VL-32B-Instruct.csv")
instr2 = pd.read_csv("./PipelineB-Answers/2nd_Qwen3-VL-32B-Instruct.csv")

In [51]:
dfs = [think1, think2, moe1, moe2, instr1, instr2]

for df in dfs:
    df["scope"].replace({
        "scope_1": "1",
        "scope_2_market_based": "2mb",
        "scope_2_location_based": "2lb",
        "scope_3": "3"
    }, inplace=True)

/var/folders/bm/8n04mnqj1sx13v8gt1bn2q7r0000gn/T/ipykernel_50123/1528221818.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["scope"].replace({
/var/folders/bm/8n04mnqj1sx13v8gt1bn2q7r0000gn/T/ipykernel_50123/1528221818.py:4: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignm

Chechking if all `report_names` match without issues.

How: `indicator=True`lables column "_merge" to "both" if there is a match, else points to the source of the merge

In [43]:
merged_t = pd.merge(think1, think2, on='report_name', how='outer', suffixes=("t1","t2"), indicator=True)
print("OK") if (len(merged_t[merged_t['_merge'] != 'both']) == 0) else print("NOK THINK")

merged_m = pd.merge(moe1, moe2, on='report_name', how='outer', suffixes=("m1","m2"), indicator=True)
print("OK") if (len(merged_m[merged_m['_merge'] != 'both']) == 0) else print("NOK MOE")

merged_i = pd.merge(instr1, instr2, on='report_name', how='outer', suffixes=("i1","i2"), indicator=True)
print("OK") if (len(merged_i[merged_i['_merge'] != 'both']) == 0) else print("NOK INSTR")

OK
OK
OK


In [49]:
merged_m["scopem1"].unique()

<StringArray>
['scope_1', 'scope_2_market_based', 'scope_2_location_based', 'scope_3']
Length: 4, dtype: str

In [45]:
merged_t_m = pd.merge(
    merged_t,
    merged_m,
    how="outer"
)

merged_t_m.head()

,report_name,scopet1,yeart1,valuet1,unitt1,labelt1,scopet2,yeart2,valuet2,unitt2,...,scopem1,yearm1,valuem1,unitm1,labelm1,scopem2,yearm2,valuem2,unitm2,labelm2
0,Allianz_2022_report,scope_1,2019,42.0,ktCO2eq,Scope 1,scope_1,2019,42.0,ktCO2eq,...,scope_1,2022,30953.0,t CO2e,Direct GHG emissions,scope_1,2020,28714.0,t CO2e,Operational
1,Allianz_2022_report,scope_1,2019,42.0,ktCO2eq,Scope 1,scope_1,2019,42.0,ktCO2eq,...,scope_1,2022,30953.0,t CO2e,Direct GHG emissions,scope_1,2021,28699.0,t CO2e,Operational
2,Allianz_2022_report,scope_1,2019,42.0,ktCO2eq,Scope 1,scope_1,2019,42.0,ktCO2eq,...,scope_1,2022,30953.0,t CO2e,Direct GHG emissions,scope_1,2022,30953.0,t CO2e,Operational
3,Allianz_2022_report,scope_1,2019,42.0,ktCO2eq,Scope 1,scope_1,2019,42.0,ktCO2eq,...,scope_1,2022,30953.0,t CO2e,Direct GHG emissions,scope_2_market_based,2020,100722.0,t CO2e,Operational
4,Allianz_2022_report,scope_1,2019,42.0,ktCO2eq,Scope 1,scope_1,2019,42.0,ktCO2eq,...,scope_1,2022,30953.0,t CO2e,Direct GHG emissions,scope_2_market_based,2021,54689.0,t CO2e,Operational


In [ ]:
# Replace scope values in all DataFrames
dfs = [think1, think2, moe1, moe2, instr1, instr2]
replacement_map = {
    '1': 'scope_1',
    '2lb': 'scope_2_location_based',
    '2mb': 'scope_2_market_based',
    '3': 'scope_3'
}

for df in dfs:
    df['scope'] = df['scope'].replace(replacement_map)